FRANS HOFFMANN

ML441 ASSIGNMENT 3

OPTION 6: 

In [ ]:
#
#imports
from copy import deepcopy
from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

HERE = Path.cwd()
ROOT = HERE if (HERE / "data").is_dir() else HERE.parent
CODE_DIR = HERE if (HERE / "minority_first_adaptive_growth.py").is_file() else ROOT / "cleaned_code"
sys.path.insert(0, str(CODE_DIR))
from minority_first_adaptive_growth import (
    Budget, Config, Data, DirectHiddenNet, fit_adaptive,
    fit_conventional, metrics, train_segment,
)

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
np.set_printoptions(precision=4, suppress=True)
print("Project:", ROOT)
print("Python / PyTorch:", sys.version.split()[0], torch.__version__)
print("scikit-learn:", __import__("sklearn").__version__)

Vehicle diagnostics:

In [ ]:
vehicle = pd.read_csv(ROOT / "data" / "vehicle_silhouettes.csv")
vehicle_features = vehicle.columns[:-1].tolist()
labels = vehicle["class"]
idx = np.arange(len(vehicle))
tr_pool, hold = train_test_split(idx, train_size=.60, random_state=42, stratify=labels)
va_idx, unused_test = train_test_split(hold, test_size=.50, random_state=42, stratify=labels.iloc[hold])
rng = np.random.default_rng(42)
target = {"van":40,"opel":65,"saab":90,"bus":115}
selected=[]
for name,n in target.items():
    eligible=tr_pool[labels.iloc[tr_pool].to_numpy()==name]
    selected.extend(rng.choice(eligible,n,replace=False))
selected=np.asarray(selected,dtype=int)
order=sorted(target,key=lambda c:(int((labels.iloc[selected]==c).sum()),c))
vm=StandardScaler().fit(vehicle.loc[selected,vehicle_features])
VX=torch.tensor(vm.transform(vehicle.loc[selected,vehicle_features]),dtype=torch.float32)
VY=torch.tensor(pd.Categorical(vehicle.loc[selected,"class"],categories=order,ordered=True).codes,dtype=torch.long)
VXV=torch.tensor(vm.transform(vehicle.loc[va_idx,vehicle_features]),dtype=torch.float32)
VYV=torch.tensor(pd.Categorical(vehicle.loc[va_idx,"class"],categories=order,ordered=True).codes,dtype=torch.long)
print("Vehicle rows:",len(vehicle),"selected training:",len(selected),"validation:",len(va_idx),"unused test:",len(unused_test))
display(pd.DataFrame({"training":pd.Series(vehicle.loc[selected,"class"]).value_counts(),"validation":vehicle.loc[va_idx,"class"].value_counts()}).reindex(order))

# One uninterrupted trajectory; retain the 40- and 120-update checkpoints.
torch.manual_seed(42)
vm0=DirectHiddenNet(len(vehicle_features),2,0)
mask=VY<2; vmask=VYV<2
budget=Budget(replace(Config(),max_updates=200,max_examples=200*int(mask.sum()),stage_updates=200))
curves=[]
trace=train_segment(vm0,(VX[mask],VY[mask],VXV[vmask],VYV[vmask]),replace(Config(),max_updates=200,max_examples=200*int(mask.sum()),stage_updates=200),budget,"vehicle_initial_2class",curves,epochs=120,adaptive=False)
vehicle_diagnostic=pd.DataFrame([dict(update=r["epoch"],validation_loss=r["validation_loss"],validation_ba=r["validation_ba"],validation_macro_f1=r["validation_f1"]) for r in curves if r["epoch"] in (40,120)])
display(vehicle_diagnostic)
print("Matched initial state; updates 40 and 120 from the same optimizer trajectory.")

Dry Bean data validation diagnostics

In [ ]:
#The dataset is deduplicated before a fixed 60/20/20 stratified split (seed 42)
# and the scaler is fitted on training rows only. The default A/B/C comparison uses
# shared controller defaults; the growth sensitivity reuses its A baseline and runs B/C with the declared 0.94 target and A-derived width ceiling. An independent unweighted logistic model is a linear reference. No Bean test scores are computed.

bean_raw=pd.read_csv(ROOT/"data"/"candidate_sources"/"uci_dry_bean.csv")
bean_features=bean_raw.columns.drop("Class").tolist()
assert not bean_raw.groupby(bean_features)["Class"].nunique().gt(1).any()
bean=bean_raw.drop_duplicates().copy()
btr,bhold=train_test_split(bean.index.to_numpy(),test_size=.4,random_state=42,stratify=bean["Class"])
bval,bunused=train_test_split(bhold,test_size=.5,random_state=42,stratify=bean.loc[bhold,"Class"])
bcounts = bean.loc[btr, "Class"].value_counts()
border = sorted(bcounts.index, key=lambda c: (bcounts[c], c))
bmap = {class_name: index for index, class_name in enumerate(border)}
bsc = StandardScaler().fit(bean.loc[btr, bean_features])
bdata = Data(
    torch.tensor(bsc.transform(bean.loc[btr, bean_features]), dtype=torch.float32),
    torch.tensor(bean.loc[btr, "Class"].map(bmap).to_numpy(), dtype=torch.long),
    torch.tensor(bsc.transform(bean.loc[bval, bean_features]), dtype=torch.float32),
    torch.tensor(bean.loc[bval, "Class"].map(bmap).to_numpy(), dtype=torch.long),
)
assert (len(btr),len(bval),len(bunused))==(8125,2709,2709)
print("Rows:",len(bean_raw),"->",len(bean),"; split:",len(btr),len(bval),len(bunused),"; ordered classes:",border)
display(pd.DataFrame({"train":bcounts,"validation":bean.loc[bval,"Class"].value_counts()}).loc[border])

# Independent logistic reference, standardized on training only.
with threadpool_limits(limits=1):
    blog = LogisticRegression(
        penalty=None, solver="newton-cg", tol=1e-6, max_iter=1000
    ).fit(
        bsc.transform(bean.loc[btr, bean_features]), bean.loc[btr, "Class"]
    )
bpred=blog.predict(bsc.transform(bean.loc[bval,bean_features]))
bean_linear = {
    "model": "sklearn linear",
    "width": 0,
    "ba": balanced_accuracy_score(bean.loc[bval, "Class"], bpred),
    "f1": f1_score(bean.loc[bval, "Class"], bpred, average="macro"),
    "converged": int(blog.n_iter_.max()) < 1000,
    "iterations": int(blog.n_iter_.max()),
}

# Default A/B/C reference (seed 42), followed by independently declared growth sensitivity.
def summarize_bean_run(name, result):
    summary = result["summary"]
    return dict(
        model=name,
        width=summary["width"],
        classes=summary["classes"],
        updates=summary["updates"],
        examples=summary["examples"],
        completed=summary["completed_protocol"],
        target_met=summary["target_met"],
        stop_reason=summary.get("stop_reason", summary.get("reason")),
        validation_ba=summary["ba"],
        validation_macro_f1=summary["f1"],
        accepted_growths=sum(
            decision.get("event") == "grow_supported_by_probe"
            for decision in result.get("decisions", [])
        ),
    )

def run_bean_sensitivity(cfg, baseline_a):
    ceiling = baseline_a["summary"]["width"]
    adaptive_cfg = replace(cfg, max_width=ceiling)
    b = fit_adaptive(bdata, "B", adaptive_cfg, 42)
    c = fit_adaptive(bdata, "C", adaptive_cfg, 42)
    rows = [summarize_bean_run("A", baseline_a)]
    rows.extend(summarize_bean_run(name, result) for name, result in (("B", b), ("C", c)))
    return pd.DataFrame(rows), {"A": baseline_a, "B": b, "C": c}, ceiling

bean_default_runs = {"A": fit_conventional(bdata, Config(), 42)}
bean_default_ceiling = bean_default_runs["A"]["summary"]["width"]
bean_default_cfg = replace(Config(), max_width=bean_default_ceiling)
for condition in ("B", "C"):
    bean_default_runs[condition] = fit_adaptive(bdata, condition, bean_default_cfg, 42)
bean_default = pd.DataFrame([
    summarize_bean_run(name, result)
    for name, result in bean_default_runs.items()
])
bean_sens_cfg = replace(
    Config(), target_ba=.94, recall_floor=.88, probe_epochs=200,
    growth_gain=.002, stage_updates=2400,
)
bean_sensitivity, bean_sensitivity_runs, bean_sensitivity_ceiling = run_bean_sensitivity(
    bean_sens_cfg, bean_default_runs["A"]
)
bean_summary = pd.concat(
    [bean_default.assign(protocol="default final-core"),
     bean_sensitivity.assign(protocol="growth sensitivity")],
    ignore_index=True,
)
bean_result_rows = bean_summary.copy()
display(pd.concat(
    [pd.DataFrame([bean_linear]),
     bean_summary.rename(columns={"validation_ba": "ba", "validation_macro_f1": "f1"})],
    ignore_index=True, sort=False,
))
print("A-derived ceilings:",bean_default_ceiling,bean_sensitivity_ceiling)

Letter A-F capacity screenin

In [ ]:
letter_raw=pd.read_csv(ROOT/"data"/"candidate_sources"/"uci_letter_recognition.csv")
letter_features = letter_raw.columns.drop("letter").tolist()
letter_subset = letter_raw.loc[letter_raw.letter.isin(list("ABCDEF"))]
assert not letter_subset.groupby(letter_features).letter.nunique().gt(1).any()
letter_clean = letter_subset.drop_duplicates(subset=letter_features, keep="first")
letter_dev = letter_clean.loc[letter_clean.index < 16000]
letter_test_rows = letter_clean.loc[letter_clean.index >= 16000].index.to_numpy()
lpool, lval = train_test_split(
    letter_dev.index.to_numpy(), test_size=.2, stratify=letter_dev.letter, random_state=42
)
lrng = np.random.default_rng(1042)
ltrain = []
for character, count in zip("ABCDEF", (100, 150, 200, 250, 300, 350)):
    eligible = np.sort(
        lpool[letter_raw.loc[lpool, "letter"].eq(character).to_numpy()]
    )
    ltrain.extend(lrng.choice(eligible, count, replace=False).tolist())
ltrain = np.asarray(ltrain, dtype=int)
lorder = list("ABCDEF")
lmap = {character: index for index, character in enumerate(lorder)}
lsc = StandardScaler().fit(letter_raw.loc[ltrain, letter_features])
lx = torch.tensor(
    lsc.transform(letter_raw.loc[ltrain, letter_features]), dtype=torch.float32
)
ly = torch.tensor(
    letter_raw.loc[ltrain, "letter"].map(lmap).to_numpy(), dtype=torch.long
)
lxv = torch.tensor(
    lsc.transform(letter_raw.loc[lval, letter_features]), dtype=torch.float32
)
lyv = torch.tensor(
    letter_raw.loc[lval, "letter"].map(lmap).to_numpy(), dtype=torch.long
)
ldata=Data(lx,ly,lxv,lyv)
assert (len(ltrain),len(lval),len(letter_test_rows))==(1350,718,845)
print("Clean A-F rows:",len(letter_clean),"; train/validation/reserved test:",len(ltrain),len(lval),len(letter_test_rows),"; counts:",np.bincount(ly.numpy()))

screen_cfg = replace(
    Config(), target_ba=.90, recall_floor=.80, probe_epochs=100,
    growth_gain=.005, max_width=16, stage_updates=1200,
)
screen = fit_conventional(ldata, screen_cfg, 42, widths=(0, 8, 16))
screen_rows = []
for row in screen["decisions"]:
    screen_rows.append({
        "width": row["width"],
        "updates": row["updates"],
        "examples": row["examples"],
        "validation_ba": row["ba"],
        "validation_macro_f1": row["f1"],
        "loss": row["loss"],
        "reason": row["reason"],
    })
letter_capacity_screen=pd.DataFrame(screen_rows)
letter_capacity = letter_capacity_screen
display(letter_capacity_screen)

NB MAJOR LETTER TEST

In [ ]:
SEEDS = (42, 43, 44, 45, 46)
EXPOSURE = 4_050_000
letter_cfg = replace(
    Config(), target_ba=.935, recall_floor=.88, growth_gain=.002,
    probe_epochs=200, stage_updates=2400,
)

def refine_to_exposure(result, cfg, exposure, name):
    summary = dict(result["summary"])
    if summary["condition"] == "A":
        selected = next(
            row for row in result["decisions"]
            if row["width"] == summary["width"]
        )
        summary["updates"] = selected["updates"]
        summary["examples"] = selected["examples"]
    charged = summary["examples"]
    remaining = (exposure - charged) // len(ltrain)
    refine_cfg = replace(cfg, stage_updates=cfg.max_updates, max_examples=exposure)
    budget = Budget(refine_cfg)
    budget.updates = summary["updates"]
    budget.examples = charged
    curves = []
    refined = train_segment(
        deepcopy(result["model"]), ldata.stage(ldata.classes), refine_cfg,
        budget, name + "_refinement", curves, epochs=remaining,
    )
    assert refined["epochs"] == remaining
    assert refined["reason"] == "probe_complete"
    assert 0 <= exposure - budget.examples < len(ltrain)
    return refined, budget, curves

principal = {}
principal_rows = []
principal_curves = []
for seed in SEEDS:
    a = fit_conventional(ldata, letter_cfg, seed)
    ceiling = a["summary"]["width"]
    adaptive_cfg = replace(letter_cfg, max_width=ceiling, max_examples=EXPOSURE)
    fits = {
        "A": a,
        "B": fit_adaptive(ldata, "B", adaptive_cfg, seed),
        "C": fit_adaptive(ldata, "C", adaptive_cfg, seed),
    }
    for name, result in fits.items():
        summary = result["summary"]
        assert summary["classes"] == ldata.classes, (seed, name, summary)
        if name == "A":
            candidate = next(
                row for row in result["decisions"]
                if row["width"] == summary["width"]
            )
            native_updates = candidate["updates"]
            native_examples = candidate["examples"]
            search_updates = summary["search_updates"]
            search_examples = summary["search_examples"]
        else:
            native_updates = summary["updates"]
            native_examples = summary["examples"]
            search_updates = 0
            search_examples = 0
        refined, budget, refinement_curves = refine_to_exposure(
            result, adaptive_cfg, EXPOSURE, f"{name}{seed}"
        )
        selected_model = refined["best"]["model"]
        validation = metrics(selected_model, lxv, lyv)
        refinement_examples = budget.examples - native_examples
        additional_search_examples = (
            search_examples - native_examples if name == "A" else 0
        )
        principal[(seed, name)] = {
            "model": selected_model,
            "native": result,
            "ceiling": ceiling,
            "budget": budget,
        }
        principal_rows.append({
            "seed": seed,
            "condition": name,
            "width": selected_model.width,
            "parameters": sum(parameter.numel() for parameter in selected_model.parameters()),
            "ceiling": ceiling,
            "classes": summary["classes"],
            "native_updates": native_updates,
            "native_examples": native_examples,
            "search_updates": search_updates,
            "search_examples": search_examples,
            "additional_search_examples": additional_search_examples,
            "refinement_updates": refined["epochs"],
            "refinement_examples": refinement_examples,
            "matched_updates": budget.updates,
            "matched_examples": budget.examples,
            "total_procedure_examples": budget.examples + additional_search_examples,
            "native_completed": summary["completed_protocol"],
            "native_target_met": summary["target_met"],
            "native_stop_reason": summary.get("stop_reason", summary.get("reason")),
            "accepted_growths": sum(
                decision.get("event") == "grow_supported_by_probe"
                for decision in result["decisions"]
            ),
            "validation_ba": validation["ba"],
            "validation_macro_f1": validation["f1"],
            "validation_loss": validation["loss"],
        })
        principal_curves.extend(
            dict(seed=seed, condition=name, phase="native", **row)
            for row in result["curves"]
        )
        principal_curves.extend(
            dict(seed=seed, condition=name, phase="refinement", **row)
            for row in refinement_curves
        )
    print(f"Principal seed {seed} complete.")

principal_fitting_results = pd.DataFrame(principal_rows)
assert len(principal_fitting_results) == 15
assert principal_fitting_results["classes"].eq(6).all()
assert (principal_fitting_results["matched_examples"] <= EXPOSURE).all()
assert (EXPOSURE - principal_fitting_results["matched_examples"] < len(ltrain)).all()
display(principal_fitting_results)

# All models and validation-selected checkpoints are frozen before test access.
letter_test_x = torch.tensor(
    lsc.transform(letter_raw.loc[letter_test_rows, letter_features]), dtype=torch.float32
)
letter_test_y = letter_raw.loc[letter_test_rows, "letter"].map(lmap).to_numpy()
test_rows = []
class_rows = []
prediction_rows = []
for (seed, name), item in principal.items():
    model = item["model"]
    with torch.no_grad():
        prediction = model(letter_test_x).argmax(1).numpy()
    recalls = recall_score(
        letter_test_y, prediction, labels=np.arange(6), average=None, zero_division=0
    )
    class_f1 = f1_score(
        letter_test_y, prediction, labels=np.arange(6), average=None, zero_division=0
    )
    predicted_counts = np.bincount(prediction, minlength=6)
    test_rows.append({
        "seed": seed,
        "condition": name,
        "width": model.width,
        "macro_f1": f1_score(letter_test_y, prediction, average="macro"),
        "accuracy": accuracy_score(letter_test_y, prediction),
        "balanced_accuracy": balanced_accuracy_score(letter_test_y, prediction),
        "minority_recall": float(recalls[:2].mean()),
        "predicted_counts": predicted_counts.tolist(),
    })
    for index, character in enumerate(lorder):
        class_rows.append({
            "seed": seed,
            "condition": name,
            "class_name": character,
            "recall": recalls[index],
            "f1": class_f1[index],
            "support": int((letter_test_y == index).sum()),
            "predicted_count": int(predicted_counts[index]),
        })
    prediction_rows.extend(zip([seed] * len(prediction), [name] * len(prediction), prediction))
letter_test = pd.DataFrame(test_rows)
letter_class = pd.DataFrame(class_rows)
principal_test_results = letter_test.copy()
principal_test_classwise = letter_class.copy()
principal_test_predictions = pd.DataFrame(
    prediction_rowss, columns=["seed", "condition", "predicted_label"]
)
display(letter_test.sort_values(["seed", "condition"]))
headline = letter_test.groupby("condition")[[
    "macro_f1", "accuracy", "balanced_accuracy"
]].agg(["mean", "std"])
display(headline)
paired_macro_f1 = letter_test.pivot(
    index="seed", columns="condition", values="macro_f1"
)
display(pd.DataFrame({"B minus A": paired_macro_f1.B - paired_macro_f1.A,
                      "C minus A": paired_macro_f1.C - paired_macro_f1.A}))
display(letter_class.pivot_table(
    index="class_name", columns="condition", values="recall", aggfunc="mean"
).reindex(lorder))

# Logical report figure 1: direct linear path plus optional trainable hidden path.
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.axis("off")
for xy, label in (((.12, .5), "Input features"),
                  ((.50, .5), "Optional tanh hidden layer"),
                  ((.88, .5), "Class logits")):
    ax.text(*xy, label, ha="center", va="center", zorder=5,
            bbox=dict(boxstyle="round", fc="white", ec="0.3"))
ax.annotate("", xy=(.79, .5), xytext=(.23, .5), zorder=1,
            arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=-.30"))
ax.annotate("", xy=(.32, .5), xytext=(.23, .5), zorder=1,
            arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(.79, .5), xytext=(.68, .5), zorder=1,
            arrowprops=dict(arrowstyle="->"))
ax.text(.50, .90, "Direct linear bypass", ha="center", zorder=5)
ax.set_title("Shared model: direct path plus optional trainable hidden path")
plt.show()

# Logical report figure 2: paired test macro F1 and classwise recall.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for seed in SEEDS:
    seed_scores = letter_test.loc[letter_test.seed == seed].set_index("condition")
    axes[0].plot(
        [0, 1, 2], [seed_scores.loc[name, "macro_f1"] for name in ("A", "B", "C")],
        color="0.75", linewidth=.8, marker=".", zorder=1,
    )
for x, name in enumerate(("A", "B", "C")):
    group = letter_test.loc[letter_test.condition == name, "macro_f1"]
    axes[0].errorbar(
        x, group.mean(), yerr=group.std(ddof=1), marker="o", capsize=4,
        color=f"C{x}", zorder=2,
    )
axes[0].set(xticks=[0,1,2], xticklabels=["A", "B", "C"], ylabel="Test macro F1",
             title="Mean and sample SD across paired seeds")
axes[0].grid(axis="y", alpha=.25)
for name, group in letter_class.groupby("condition"):
    recall_stats = group.groupby("class_name").recall.agg(["mean", "std"]).reindex(lorder)
    axes[1].errorbar(lorder, recall_stats["mean"], yerr=recall_stats["std"],
                     marker="o", capsize=3, label=name)
axes[1].set(xlabel="Class; training counts 100, 150, 200, 250, 300, 350",
             ylabel="Test recall", ylim=(0, 1.02), title="Classwise mean and sample SD")
axes[1].grid(axis="y", alpha=.25)
axes[1].legend(title="Condition")
fig.tight_layout()
plt.show()

plots

In [ ]:
SENS_SEEDS = (42, 43, 44)
SENS_EXPOSURE = 8_100_000
reference_cfg = letter_cfg
tuned_cfg = replace(
    reference_cfg, target_ba=.96, probe_epochs=600, growth_gain=.0015,
    stage_updates=4800, max_probes=8,
)
sensitivity_rows = []
sensitivity_decision_rows = []
sensitivity_class_rows = []
sensitivity_native = {}

for seed in SENS_SEEDS:
    baseline = fit_conventional(ldata, reference_cfg, seed)
    ceiling = baseline["summary"]["width"]
    selected_a = next(
        row for row in baseline["decisions"] if row["width"] == ceiling
    )
    baseline_for_refinement = dict(baseline)
    baseline_for_refinement["summary"] = dict(baseline["summary"])
    baseline_for_refinement["summary"].update(
        updates=selected_a["updates"], examples=selected_a["examples"]
    )
    baseline_cfg = replace(reference_cfg, max_width=ceiling, max_updates=18000)
    baseline_refined, baseline_budget, _ = refine_to_exposure(
        baseline_for_refinement, baseline_cfg, SENS_EXPOSURE, f"sensitivity_A_{seed}"
    )
    baseline_model = baseline_refined["best"]["model"]
    baseline_score = metrics(baseline_model, lxv, lyv)
    sensitivity_rows.append({
        "seed": seed, "package": "reference", "condition": "A",
        "classes": ldata.classes, "full_class": True, "width": baseline_model.width,
        "ceiling": ceiling, "native_completed": baseline["summary"]["completed_protocol"],
        "native_target_met": baseline["summary"]["target_met"],
        "native_stop_reason": baseline["summary"].get(
            "stop_reason", baseline["summary"].get("reason")
        ),
        "native_updates": selected_a["updates"],
        "native_examples": selected_a["examples"],
        "matched_updates": baseline_budget.updates,
        "matched_examples": baseline_budget.examples,
        "validation_macro_f1": baseline_score["f1"],
        "validation_ba": baseline_score["ba"],
        "minority_recall": float(np.mean(baseline_score["recall"][:2])),
    })
    for package, config in (("reference", reference_cfg), ("tuned", tuned_cfg)):
        adaptive_cfg = replace(
            config, max_width=ceiling, max_examples=SENS_EXPOSURE, max_updates=18000
        )
        for condition in ("B", "C"):
            result = fit_adaptive(ldata, condition, adaptive_cfg, seed)
            summary = result["summary"]
            sensitivity_native[(seed, package, condition)] = result
            sensitivity_decision_rows.extend(
                dict(seed=seed, package=package, **decision)
                for decision in result["decisions"]
            )
            row = {
                "seed": seed, "package": package, "condition": condition,
                "classes": summary["classes"],
                "full_class": summary["classes"] == ldata.classes,
                "width": result["model"].width, "ceiling": ceiling,
                "native_completed": summary["completed_protocol"],
                "native_target_met": summary["target_met"],
                "native_stop_reason": summary["stop_reason"],
                "native_updates": summary["updates"],
                "native_examples": summary["examples"],
                "matched_updates": summary["updates"],
                "matched_examples": summary["examples"],
                "validation_macro_f1": np.nan,
                "validation_ba": np.nan,
                "minority_recall": np.nan,
            }
            if summary["classes"] == ldata.classes:
                refined, budget, _ = refine_to_exposure(
                    result, adaptive_cfg, SENS_EXPOSURE,
                    f"sensitivity_{seed}_{package}_{condition}",
                )
                final_model = refined["best"]["model"]
                final_score = metrics(final_model, lxv, lyv)
                assert 0 <= SENS_EXPOSURE - budget.examples < len(ltrain)
                row.update(
                    width=final_model.width,
                    matched_updates=budget.updates,
                    matched_examples=budget.examples,
                    validation_macro_f1=final_score["f1"],
                    validation_ba=final_score["ba"],
                    minority_recall=float(np.mean(final_score["recall"][:2])),
                )
                for index, character in enumerate(lorder):
                    sensitivity_class_rows.append({
                        "seed": seed, "package": package, "condition": condition,
                        "class_name": character,
                        "recall": final_score["recall"][index],
                        "f1": final_score["class_f1"][index],
                        "support": final_score["support"][index],
                    })
            sensitivity_rows.append(row)
    print(f"Letter sensitivity seed {seed} complete.")

sensitivity_fitting_results = pd.DataFrame(sensitivity_rows)
sensitivity_validation_results = sensitivity_fitting_results[[
    "seed", "package", "condition", "classes", "full_class", "width",
    "validation_macro_f1", "validation_ba", "minority_recall",
]].copy()
sensitivity_decisions = pd.DataFrame(sensitivity_decision_rows)
sensitivity_classwise_validation = pd.DataFrame(sensitivity_class_rows)
display(sensitivity_fitting_results.sort_values(["seed", "condition", "package"]))
display(sensitivity_classwise_validation)
display(sensitivity_decisions)

# Logical report figure 3: retained widths and full-class validation macro F1.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
seed_colors = {42: "tab:blue", 43: "tab:orange", 44: "tab:green"}
plot_items = [
    ("A reference", "A", "reference", 0),
    ("B reference", "B", "reference", 1),
    ("C reference", "C", "reference", 2),
    ("B tuned", "B", "tuned", 3),
    ("C tuned", "C", "tuned", 4),
]
for position, (label, condition, package, _) in enumerate(plot_items):
    rows = sensitivity_fitting_results[
        (sensitivity_fitting_results["condition"] == condition)
        & (sensitivity_fitting_results["package"] == package)
    ]
    for _, row in rows.iterrows():
        color = seed_colors[row["seed"]]
        marker = "o" if package == "reference" else "s"
        width_marker = marker if bool(row["full_class"]) else "x"
        axes[0].scatter(position, row["width"], color=color, marker=width_marker)
        if bool(row["full_class"]):
            axes[1].scatter(position, row["validation_macro_f1"],
                            color=color, marker=marker)
axes[0].set_title("Retained hidden width")
axes[0].set_ylabel("Hidden units")
axes[0].set_xticks(range(len(plot_items)), [item[0] for item in plot_items], rotation=25)
axes[0].grid(axis="y", alpha=.25)
axes[1].set_title("Full-class validation macro F1")
axes[1].set_ylabel("Validation macro F1")
axes[1].set_xticks(range(len(plot_items)), [item[0] for item in plot_items], rotation=25)
axes[1].set_ylim(.90, .99)
axes[1].grid(axis="y", alpha=.25)
for position, (_, condition, package, _) in enumerate(plot_items):
    if condition == "A":
        continue
    denominator = sensitivity_fitting_results.loc[
        (sensitivity_fitting_results["condition"] == condition)
        & (sensitivity_fitting_results["package"] == package), "full_class"
    ].sum()
    axes[1].text(position, .901, f"{int(denominator)}/3", ha="center", va="bottom", fontsize=8)
seed_handles = [
    axes[0].scatter([], [], color=color, marker="o", label=str(seed))
    for seed, color in seed_colors.items()
]
axes[0].legend(handles=seed_handles, title="Seed", fontsize=8)
fig.tight_layout()
plt.show()
